In [1]:
import json
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("us_recipies_Iceberg_Silver")
    .master("local[*]")
    # Added iceberg-aws-bundle to provide the Glue client dependencies
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.iceberg:iceberg-aws-bundle:1.4.3")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    
    # Configure Iceberg to use AWS Glue Data Catalog
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://us-recipies-bronze-v1/iceberg-warehouse/")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    
    # AWS S3 Timeout overrides
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.socket.timeout", "60000")
    .getOrCreate()
)

# Route Spark's internal Hadoop connection to AWS S3
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.profile.ProfileCredentialsProvider")
hadoop_conf.set("fs.s3a.endpoint", "s3.amazonaws.com")

In [2]:
spark.sql('show databases in glue_catalog').show()

+--------------+
|     namespace|
+--------------+
|bronze_recipes|
|       default|
+--------------+



In [5]:
spark.sql('show tables in glue_catalog.bronze_recipes').show(truncate=False)

+--------------+-----------------------+-----------+
|namespace     |tableName              |isTemporary|
+--------------+-----------------------+-----------+
|bronze_recipes|recipes_unpacked_silver|false      |
|bronze_recipes|us_raw                 |false      |
+--------------+-----------------------+-----------+



In [3]:
df_raw = spark.read.format('iceberg').load('glue_catalog.bronze_recipes.us_raw')

In [8]:
#lets create silver layers
spark.sql('CREATE NAMESPACE  IF NOT EXISTS glue_catalog.silver_recipes')

DataFrame[]

In [10]:
spark.sql('show databases in glue_catalog').show()

+--------------+
|     namespace|
+--------------+
|bronze_recipes|
|       default|
|silver_recipes|
+--------------+



In [10]:
from pyspark.sql import functions as f


In [21]:
df_raw.columns

['Contient',
 'Country_State',
 'URL',
 'cook_time',
 'cuisine',
 'description',
 'ingredients',
 'instructions',
 'nutrients_legacy_string',
 'prep_time',
 'rating',
 'serves',
 'title',
 'total_time',
 'nutrients']

In [4]:


df_ingredients = df_raw.select(
    f.col('title'),
    f.explode(f.col('ingredients')).alias('ingredients')
)
#.limit(5).show(truncate=False)

In [5]:
df_ingredients \
    .writeTo('glue_catalog.silver_recipes.recipes_ingredients') \
    .using('iceberg') \
    .createOrReplace()
    #.partitionedBy(f.bucket(32,'title')) \


In [8]:
spark.sql('show tables in glue_catalog.silver_recipes').show(truncate=False)

+--------------+-------------------+-----------+
|namespace     |tableName          |isTemporary|
+--------------+-------------------+-----------+
|silver_recipes|recipes_ingredients|false      |
+--------------+-------------------+-----------+



In [12]:
df_silver_instructions = (
    df_raw
    .select(
        f.col("title"),
        f.explode(f.col("instructions")).alias("instruction")
    )
)


In [13]:
df_silver_instructions.writeTo('glue_catalog.silver_recipes.recipes_instructions') \
    .using('iceberg') \
    .createOrReplace()

In [14]:
spark.sql("""ALTER TABLE glue_catalog.silver_recipes.recipes_instructions
            WRITE ORDERED BY title
          """)

DataFrame[]

In [15]:
spark.sql("""ALTER TABLE glue_catalog.silver_recipes.recipes_ingredients
            WRITE ORDERED BY title
          """)

DataFrame[]

In [18]:
spark.sql("""
    CALL glue_catalog.system.rewrite_data_files(
        table=> 'glue_catalog.silver_recipes.recipes_ingredients',
        strategy=> 'sort'
    )
    """
)

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]

In [19]:
spark.sql("""
    CALL glue_catalog.system.rewrite_data_files(
        table=> 'glue_catalog.silver_recipes.recipes_instructions',
        strategy => 'sort'
    )
    """
)

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]

In [21]:
df_silver_unpacked = spark.sql('select * from glue_catalog.bronze_recipes.recipes_unpacked_silver')

In [23]:
df_silver = df_silver_unpacked.drop('ingredients','instructions').sort('title')

In [24]:
df_silver.writeTo('glue_catalog.silver_recipes.silver_recipes') \
    .using('iceberg') \
    .partitionedBy(f.bucket(20,'title')) \
    .createOrReplace()


In [25]:
spark.sql('show tables in glue_catalog.silver_recipes').show(truncate=False)

+--------------+--------------------+-----------+
|namespace     |tableName           |isTemporary|
+--------------+--------------------+-----------+
|silver_recipes|recipes_ingredients |false      |
|silver_recipes|recipes_instructions|false      |
|silver_recipes|silver_recipes      |false      |
+--------------+--------------------+-----------+

